## Simulating Ibtesam Ahmed recommendation algorithms from Kaggle

### Content-based recommender doesn't use ML, Collaborative does

#### Link: https://www.kaggle.com/code/ibtesama/getting-started-with-a-movie-recommendation-system/notebook

# Neural Content-Based Recommender
## Plot + Credits, Genres & Keywords via Sentence Transformers

In [9]:
import pandas as pd
import numpy as np

df1=pd.read_csv('../tmdb/tmdb_5000_credits.csv')
df2=pd.read_csv('../tmdb/tmdb_5000_movies.csv')

In [10]:
# Join two datasets on id column
df1.columns = ['id','tittle','cast','crew']
df2= df2.merge(df1,on='id')

In [11]:
df2.head()

,budget,genres,homepage,id,keywords,original_language,original_title,overview,popularity,production_companies,...,runtime,spoken_languages,status,tagline,title,vote_average,vote_count,tittle,cast,crew
0,237000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://www.avatarmovie.com/,19995,"[{""id"": 1463, ""name"": ""culture clash""}, {""id"":...",en,Avatar,"In the 22nd century, a paraplegic Marine is di...",150.437577,"[{""name"": ""Ingenious Film Partners"", ""id"": 289...",...,162.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}, {""iso...",Released,Enter the World of Pandora.,Avatar,7.2,11800,Avatar,"[{""cast_id"": 242, ""character"": ""Jake Sully"", ""...","[{""credit_id"": ""52fe48009251416c750aca23"", ""de..."
1,300000000,"[{""id"": 12, ""name"": ""Adventure""}, {""id"": 14, ""...",http://disney.go.com/disneypictures/pirates/,285,"[{""id"": 270, ""name"": ""ocean""}, {""id"": 726, ""na...",en,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha...",139.082615,"[{""name"": ""Walt Disney Pictures"", ""id"": 2}, {""...",...,169.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,"At the end of the world, the adventure begins.",Pirates of the Caribbean: At World's End,6.9,4500,Pirates of the Caribbean: At World's End,"[{""cast_id"": 4, ""character"": ""Captain Jack Spa...","[{""credit_id"": ""52fe4232c3a36847f800b579"", ""de..."
2,245000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://www.sonypictures.com/movies/spectre/,206647,"[{""id"": 470, ""name"": ""spy""}, {""id"": 818, ""name...",en,Spectre,A cryptic message from Bond’s past sends him o...,107.376788,"[{""name"": ""Columbia Pictures"", ""id"": 5}, {""nam...",...,148.0,"[{""iso_639_1"": ""fr"", ""name"": ""Fran\u00e7ais""},...",Released,A Plan No One Escapes,Spectre,6.3,4466,Spectre,"[{""cast_id"": 1, ""character"": ""James Bond"", ""cr...","[{""credit_id"": ""54805967c3a36829b5002c41"", ""de..."
3,250000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 80, ""nam...",http://www.thedarkknightrises.com/,49026,"[{""id"": 849, ""name"": ""dc comics""}, {""id"": 853,...",en,The Dark Knight Rises,Following the death of District Attorney Harve...,112.312950,"[{""name"": ""Legendary Pictures"", ""id"": 923}, {""...",...,165.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,The Legend Ends,The Dark Knight Rises,7.6,9106,The Dark Knight Rises,"[{""cast_id"": 2, ""character"": ""Bruce Wayne / Ba...","[{""credit_id"": ""52fe4781c3a36847f81398c3"", ""de..."
4,260000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://movies.disney.com/john-carter,49529,"[{""id"": 818, ""name"": ""based on novel""}, {""id"":...",en,John Carter,"John Carter is a war-weary, former military ca...",43.926995,"[{""name"": ""Walt Disney Pictures"", ""id"": 2}]",...,132.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,"Lost in our world, found in another.",John Carter,6.1,2124,John Carter,"[{""cast_id"": 5, ""character"": ""John Carter"", ""c...","[{""credit_id"": ""52fe479ac3a36847f813eaa3"", ""de..."


In [12]:
df2['overview'].head(5)

0    In the 22nd century, a paraplegic Marine is di...
1    Captain Barbossa, long believed to be dead, ha...
2    A cryptic message from Bond’s past sends him o...
3    Following the death of District Attorney Harve...
4    John Carter is a war-weary, former military ca...
Name: overview, dtype: str

In [13]:
# Parse the stringified features into their corresponding python objects
from ast import literal_eval

features = ['cast', 'crew', 'keywords', 'genres']
for feature in features:
    df2[feature] = df2[feature].apply(literal_eval)

#### Functions that will help extract required info from each feature

In [14]:
# Get the director's name from the crew feature. If director is not listed, return NaN
def get_director(x):
    for i in x:
        if i['job'] == 'Director':
            return i['name']
    return np.nan

In [15]:
# Test directors function

df2['director'] = df2['crew'].apply(get_director)

df2[['title', 'director']].head()

,title,director
0,Avatar,James Cameron
1,Pirates of the Caribbean: At World's End,Gore Verbinski
2,Spectre,Sam Mendes
3,The Dark Knight Rises,Christopher Nolan
4,John Carter,Andrew Stanton


In [16]:
# Returns the list top 3 elements or entire list; whichever is more.
def get_list(x):
    if isinstance(x, list):
        names = [i['name'] for i in x]
        
        # Check if more than 3 elements exist. If yes, return only first three. If no, return entire list.
        if len(names) > 3:
            names = names[:3]
        return names

    # Return empty list in case of missing/malformed data
    return []

In [17]:
# Define new director, cast, genres and keywords features that are in a suitable form.
df2['director'] = df2['crew'].apply(get_director)

features = ['cast', 'keywords', 'genres']
for feature in features:
    df2[feature] = df2[feature].apply(get_list)

In [18]:
# Print the new features of the first 3 films
df2[['title', 'cast', 'director', 'keywords', 'genres']].head(3)

,title,cast,director,keywords,genres
0,Avatar,"[Sam Worthington, Zoe Saldana, Sigourney Weaver]",James Cameron,"[culture clash, future, space war]","[Action, Adventure, Fantasy]"
1,Pirates of the Caribbean: At World's End,"[Johnny Depp, Orlando Bloom, Keira Knightley]",Gore Verbinski,"[ocean, drug abuse, exotic island]","[Adventure, Fantasy, Action]"
2,Spectre,"[Daniel Craig, Christoph Waltz, Léa Seydoux]",Sam Mendes,"[spy, based on novel, secret agent]","[Action, Adventure, Crime]"


#### The next step would be to convert the names and keyword instances into lowercase and strip all the spaces between them. This is done so that our vectorizer doesn't count the Johnny of "Johnny Depp" and "Johnny Galecki" as the same.

In [19]:
# Function to convert all strings to lower case and strip names of spaces
def clean_data(x):
    if isinstance(x, list):
        return [str.lower(i.replace(" ", "")) for i in x]
    else:
        # Check if director exists. If not, return empty string
        if isinstance(x, str):
            return str.lower(x.replace(" ", ""))
        else:
            return ''

In [20]:
# Apply clean_data function to your features.
features = ['cast', 'keywords', 'director', 'genres']

for feature in features:
    df2[feature] = df2[feature].apply(clean_data)

## Step 1: Build a unified text representation for each movie
#### We combine the plot overview with structured metadata (cast, director, keywords, genres) into one rich text string per movie. Metadata terms are repeated to give them comparable signal weight alongside the longer plot text.

In [21]:
def create_unified_text(x):
    plot     = x['overview']              if isinstance(x['overview'],  str) else ''
    cast     = ' '.join(x['cast'])        if isinstance(x['cast'],     list) else ''
    genres   = ' '.join(x['genres'])      if isinstance(x['genres'],   list) else ''
    keywords = ' '.join(x['keywords'])    if isinstance(x['keywords'], list) else ''
    director = x['director']              if isinstance(x['director'],  str) else ''
    
    # Metadata terms repeated twice to balance against the longer plot text
    return f"{plot} {cast} {cast} {director} {director} {genres} {genres} {keywords} {keywords}".strip()

df2['overview'] = df2['overview'].fillna('')
df2['unified_text'] = df2.apply(create_unified_text, axis=1)
df2['unified_text'].head(3)

0    In the 22nd century, a paraplegic Marine is di...
1    Captain Barbossa, long believed to be dead, ha...
2    A cryptic message from Bond’s past sends him o...
Name: unified_text, dtype: str

## Step 2: Generate neural embeddings with Sentence Transformers
#### The `all-MiniLM-L6-v2` model is a lightweight BERT-based neural network trained to produce semantically meaningful 384-dimensional vectors. Unlike TF-IDF or CountVectorizer, it understands meaning — so "hero saves the world" and "protagonist rescues humanity" will score as similar even with zero word overlap.

In [ ]:
#%pip install sentence-transformers


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [23]:
from sentence_transformers import SentenceTransformer

# Load pre-trained neural model
model = SentenceTransformer('all-MiniLM-L6-v2')

# Encode all movies into 384-dimensional neural embedding vectors
# show_progress_bar gives visibility since this encodes ~4800 movies
embeddings = model.encode(
    df2['unified_text'].tolist(),
    show_progress_bar=True,
    batch_size=64
)

print(f'Embeddings shape: {embeddings.shape}')

/usr/local/python/3.12.1/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:02<00:00, 47.02it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Batches: 100%|██████████| 76/76 [03:33<00:00,  2.81s/it]

Embeddings shape: (4803, 384)


## Step 3: Compute cosine similarity between all movie embeddings
#### Each movie is now a point in 384-dimensional neural space. Cosine similarity measures the angle between two points — movies with similar meaning cluster together regardless of exact word overlap.

In [24]:
from sklearn.metrics.pairwise import cosine_similarity

# Compute pairwise cosine similarity across all neural embeddings
cosine_sim_nn = cosine_similarity(embeddings, embeddings)

# Build reverse index: movie title -> dataframe index
indices = pd.Series(df2.index, index=df2['title']).drop_duplicates()

print(f'Similarity matrix shape: {cosine_sim_nn.shape}')

Similarity matrix shape: (4803, 4803)


## Step 4: Recommendation function
#### Takes a movie title and returns the 10 most similar movies based on the neural similarity matrix.

In [25]:
def get_recommendations(title):
    
    # Make title all lowsercase
    title = title.lower()

    # Get copy of indices and make them lowercase to compare to title
    indices_lower = indices.copy()
    indices_lower.index = indices_lower.index.str.lower()

    # Return message if movie is not in list
    if title not in indices_lower:
        return f"Movie '{title}' not found in database."

    # get index of title in indices
    idx = indices_lower[title]

    # Get similarity scores for this movie against all others
    sim_scores = list(enumerate(cosine_sim_nn[idx]))
    
    # Sort by score descending
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    
    # Skip index 0 (the movie itself), and print top 10
    sim_scores = sim_scores[1:11]

    movie_indices = [i[0] for i in sim_scores]
    return df2['title'].iloc[movie_indices]

In [26]:
# Year range

df2['release_date'] = pd.to_datetime(df2['release_date'], errors='coerce')
df2['year'] = df2['release_date'].dt.year

min_year = df2['year'].min()
max_year = df2['year'].max()

print(min_year, max_year)

1916.0 2017.0


## Step 5: Test the recommender

#### Test users and movies they enjoy

In [27]:
user_history = {
    # User 1: Classic gangster & crime dramas
    "user_1": [
        "The Godfather", "GoodFellas", "Scarface", "Pulp Fiction", "The Departed",
        "The Godfather: Part II", "Casino", "Donnie Brasco", "Once Upon a Time in America",
        "The Untouchables", "Road to Perdition", "Public Enemies", "Gangs of New York",
        "A History of Violence", "Eastern Promises", "The Town", "The Conformist",
        "Find Me Guilty", "Black Mass", "The Hills Have Eyes"
    ],

    # User 2: Blockbuster action & superhero movies
    "user_2": [
        "Avatar", "Titanic", "Avengers: Age of Ultron", "Guardians of the Galaxy", "Iron Man",
        "Thor", "Captain America: The First Avenger", "The Avengers", "Ant-Man",
        "The Incredible Hulk", "Captain America: Civil War", "Iron Man 2", "Iron Man 3",
        "Thor: The Dark World", "Batman Begins", "The Dark Knight", "The Dark Knight Rises",
        "Batman & Robin", "Batman Returns", "Batman v Superman: Dawn of Justice"
    ],

    # User 3: Musical & biographical movies
    "user_3": [
        "Chicago", "Moulin Rouge!", "8MM", "Amnesiac", "Grease",
        "Les Misérables", "Inception", "The Pursuit of Happyness", "The Hit List",
        "Singin' in the Rain", "The Sound of Music", "West Side Story", "Mary Poppins",
        "The Wizard of Oz", "Frozen", "Aladdin", "Cinderella", "The Nutcracker",
        "Alice in Wonderland", "The Broadway Melody"
    ],

    # User 4: Fantasy & young adult series
    "user_4": [
        "The Lord of the Rings: The Fellowship of the Ring", "The Hobbit: An Unexpected Journey",
        "The Lord of the Rings: The Two Towers", "The Lord of the Rings: The Return of the King",
        "Harry Potter and the Philosopher's Stone", "Harry Potter and the Chamber of Secrets",
        "Harry Potter and the Prisoner of Azkaban", "The Hunger Games: Catching Fire",
        "The Hunger Games: Mockingjay - Part 2", "The Twilight Saga: New Moon",
        "The Twilight Saga: Eclipse", "The Twilight Saga: Breaking Dawn - Part 2",
        "Percy Jackson: Sea of Monsters", "Percy Jackson & the Olympians: The Lightning Thief",
        "Harry Potter and the Order of the Phoenix", "The Chronicles of Narnia: The Lion, the Witch and the Wardrobe",
        "The Hobbit: The Desolation of Smaug", "The Hobbit: The Battle of the Five Armies",
        "The Adventures of Huck Finn", "Hellboy II: The Golden Army"
    ],

    # User 5: Horror & thriller movies
    "user_5": [
        "The Shining", "1408", "8 Days", "The Conjuring", "Insidious",
        "Sinister", "Annabelle", "Paranormal Activity 2", "Halloween: Resurrection", "Psycho",
        "Jaws", "Saw: The Final Chapter", "Scream 3", "Pet Sematary", "White Noise 2: The Light",
        "It Follows", "The Possession", "The Exorcist", "Evil Dead", "Restoration"
    ]
}

In [28]:
# Test get_recommendations for every liked movie of each user

counter = 0

for user, movies in user_history.items():
    print(f"\nRecommendations for {user}:")
    for movie in movies:
        #print(f"\nMovie: {movie}")
        recs = get_recommendations(movie)
        if "not found" in recs:
            counter +=1
            print(recs)
            
print(f"\nTotal movies not found: {counter}")


Recommendations for user_1:

Recommendations for user_2:

Recommendations for user_3:

Recommendations for user_4:

Recommendations for user_5:

Total movies not found: 0


In [29]:
import random

In [53]:
# Split movies into training and testing to evaluate model
train_test_split = {}
split_ratio = 5

# list and dict to store each unknown movie for each user
users = {}

for user, movies in user_history.items():
    unknown_movies = []
    known = random.sample(movies, split_ratio)  # movies user has "rated" / known
    unknown = [m for m in movies if m not in known]  # remaining movies for evaluation
    train_test_split[user] = {"known": known, "unknown": unknown}
    unknown_movies.append(unknown)
    users[user] = unknown_movies

# Example output:
for user, split in train_test_split.items():
    print(f"{user}:")
    print("Known:", split["known"])
    print("Unknown:", split["unknown"])
    print()

user_1:
Known: ['Black Mass', 'Once Upon a Time in America', 'Pulp Fiction', 'The Town', 'Donnie Brasco']
Unknown: ['The Godfather', 'GoodFellas', 'Scarface', 'The Departed', 'The Godfather: Part II', 'Casino', 'The Untouchables', 'Road to Perdition', 'Public Enemies', 'Gangs of New York', 'A History of Violence', 'Eastern Promises', 'The Conformist', 'Find Me Guilty', 'The Hills Have Eyes']

user_2:
Known: ['Titanic', 'The Dark Knight', 'Iron Man', 'Avatar', 'Ant-Man']
Unknown: ['Avengers: Age of Ultron', 'Guardians of the Galaxy', 'Thor', 'Captain America: The First Avenger', 'The Avengers', 'The Incredible Hulk', 'Captain America: Civil War', 'Iron Man 2', 'Iron Man 3', 'Thor: The Dark World', 'Batman Begins', 'The Dark Knight Rises', 'Batman & Robin', 'Batman Returns', 'Batman v Superman: Dawn of Justice']

user_3:
Known: ["Singin' in the Rain", 'The Sound of Music', 'The Nutcracker', 'Les Misérables', 'Aladdin']
Unknown: ['Chicago', 'Moulin Rouge!', '8MM', 'Amnesiac', 'Grease', 'I

In [58]:
for key,val in users.items():
    print(f"User: {key}")
    print(f"List of movies: {val}")
    print()
    print(val[0])
    print()

User: user_1
List of movies: [['The Godfather', 'GoodFellas', 'Scarface', 'The Departed', 'The Godfather: Part II', 'Casino', 'The Untouchables', 'Road to Perdition', 'Public Enemies', 'Gangs of New York', 'A History of Violence', 'Eastern Promises', 'The Conformist', 'Find Me Guilty', 'The Hills Have Eyes']]

['The Godfather', 'GoodFellas', 'Scarface', 'The Departed', 'The Godfather: Part II', 'Casino', 'The Untouchables', 'Road to Perdition', 'Public Enemies', 'Gangs of New York', 'A History of Violence', 'Eastern Promises', 'The Conformist', 'Find Me Guilty', 'The Hills Have Eyes']

User: user_2
List of movies: [['Avengers: Age of Ultron', 'Guardians of the Galaxy', 'Thor', 'Captain America: The First Avenger', 'The Avengers', 'The Incredible Hulk', 'Captain America: Civil War', 'Iron Man 2', 'Iron Man 3', 'Thor: The Dark World', 'Batman Begins', 'The Dark Knight Rises', 'Batman & Robin', 'Batman Returns', 'Batman v Superman: Dawn of Justice']]

['Avengers: Age of Ultron', 'Guardian

In [ ]:
for user, known_movies in train_test_split.items():
    print(user)
    for key, movies in known_movies.items():
        if key == "known":
            for movie in movies:
                print(movie)
                recommended_movies = get_recommendations(movie)
                
                if recommended_movies in #list of unknow movies

user_1
Eastern Promises
Find Me Guilty
The Untouchables
Donnie Brasco
Road to Perdition
user_2
Ant-Man
Batman Begins
The Dark Knight Rises
Iron Man
Captain America: Civil War
user_3
Singin' in the Rain
Grease
Cinderella
The Pursuit of Happyness
Mary Poppins
user_4
The Chronicles of Narnia: The Lion, the Witch and the Wardrobe
The Twilight Saga: New Moon
The Hobbit: The Battle of the Five Armies
The Lord of the Rings: The Return of the King
The Lord of the Rings: The Two Towers
user_5
Paranormal Activity 2
Restoration
Saw: The Final Chapter
Halloween: Resurrection
Scream 3


In [39]:
# Generate Recommendations
for user, known_movies in train_test_split.items():
    for known_movie in known_movies:
        recommended_movies = get_recommendations(known_movie)
        print()
        print(f"This is known movie {known_movie}")
        print()
        print(f"Recommended movies for {known_movie}: {recommended_movies}")
        print()

# Feed known movies into recommender

# Check how many movies in the unknown list appear in the recommendation (hit rate)


This is known movie known

Recommended movies for known: Movie 'known' not found in database.


This is known movie unknown

Recommended movies for unknown: 4492                After
930              Non-Stop
3014        The Dead Zone
4137     Grave Encounters
2118          Premonition
478              Daylight
3717           After.Life
322     The Fifth Element
1379             The Cell
1808            Self/less
Name: title, dtype: str


This is known movie known

Recommended movies for known: Movie 'known' not found in database.


This is known movie unknown

Recommended movies for unknown: 4492                After
930              Non-Stop
3014        The Dead Zone
4137     Grave Encounters
2118          Premonition
478              Daylight
3717           After.Life
322     The Fifth Element
1379             The Cell
1808            Self/less
Name: title, dtype: str


This is known movie known

Recommended movies for known: Movie 'known' not found in database.


This is known mov

In [ ]:
# Evaluate

# Precision@K: How many recommended movies were actually liked (in unknown set).
# Recall@k: How many of the unknown liked movies did the model manage to recommend.

In [ ]:
get_recommendations('The Dark Knight Rises')

In [ ]:
get_recommendations('The Godfather')

In [ ]:
get_recommendations('The Avengers')